[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/blhuillier/2026B_Galactic_Dynamics/blob/main/notebooks/chapter_03/nb_03_classical_mechanics.ipynb)

# Chapter 3 - Elements of Classical Mechanics

**Galactic Dynamics · Sejong University · 2026B**

This notebook accompanies the three numerical exercises of Chapter 3:

| Part | Exercise | Topic |
|------|----------|-------|
| A | 3.10 | Symplectic versus non-symplectic integration |
| B | 3.11 | Rebuilding the pendulum phase portrait |
| C | 3.12 | Frequency ratios and closed orbits |

Throughout we use the scaled pendulum Hamiltonian

$$ H(\theta, p) = \tfrac12 p^2 - \cos\theta , $$

which is the system drawn in Figure 3.2 of the notes. Time is measured in
units of $\sqrt{\ell/g}$, so the small-oscillation frequency is
$\omega_0 = 1$ and the separatrix sits at $H = 1$.

Cells marked **TODO** are yours to complete. Everything else is given.

> **Running on Google Colab?** This notebook uses the `galdyn` toolkit from the
> course repository. Run the cell below first - it clones the repo and adds it
> to your Python path. On a local install (via `uv`), this cell does nothing.

In [ ]:
import os
import sys

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if not os.path.exists("2026B_Galactic_Dynamics"):
        !git clone -q https://github.com/blhuillier/2026B_Galactic_Dynamics.git
    sys.path.insert(0, "2026B_Galactic_Dynamics/src")
    %pip install -q astropy

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from galdyn.plotting import set_latex_style

set_latex_style(use_tex=False)

rng = np.random.default_rng(20268)

---
## Part A - Exercise 3.10: symplectic versus non-symplectic integration

The pendulum in first-order form is

$$ \dot\theta = p, \qquad \dot p = -\sin\theta . $$

Both integrators below take the same right-hand side and the same step, so any
difference between them is structural, not a matter of accuracy.

In [ ]:
def pendulum_rhs(state):
    """Return d(theta, p)/dt for H = p^2/2 - cos(theta)."""
    theta, p = state
    return np.array([p, -np.sin(theta)])


def pendulum_energy(theta, p):
    """Return H = p^2/2 - cos(theta)."""
    return 0.5 * p**2 - np.cos(theta)


def rk4(state0, dt, n_steps):
    """Classical fourth-order Runge-Kutta. Not symplectic."""
    out = np.empty((n_steps + 1, 2))
    out[0] = state0
    for n in range(n_steps):
        y = out[n]
        k1 = pendulum_rhs(y)
        k2 = pendulum_rhs(y + 0.5 * dt * k1)
        k3 = pendulum_rhs(y + 0.5 * dt * k2)
        k4 = pendulum_rhs(y + dt * k3)
        out[n + 1] = y + dt * (k1 + 2 * k2 + 2 * k3 + k4) / 6.0
    return out


def leapfrog(state0, dt, n_steps):
    """Second-order kick-drift-kick leapfrog. Symplectic."""
    out = np.empty((n_steps + 1, 2))
    out[0] = state0
    theta, p = state0
    for n in range(n_steps):
        p_half = p - 0.5 * dt * np.sin(theta)
        theta = theta + dt * p_half
        p = p_half - 0.5 * dt * np.sin(theta)
        out[n + 1] = (theta, p)
    return out

In [ ]:
state0 = np.array([1.0, 0.0])
dt = 0.05
n_periods = 10_000
n_steps = int(n_periods * 2 * np.pi / dt)

traj_rk4 = rk4(state0, dt, n_steps)
traj_lf = leapfrog(state0, dt, n_steps)
t = dt * np.arange(n_steps + 1)

print(f"{n_steps:,} steps, t_max = {t[-1]:.3g}")

**TODO A.1** - Plot $|H(t) - H(0)|$ against $t$ on logarithmic axes for both
integrators. Which is more accurate over one period? Over $10^4$? Describe the
*shape* of each curve, not just its size.

In [ ]:
# TODO: compute the energy error for each trajectory and plot it.
#
#   err_rk4 = np.abs(pendulum_energy(*traj_rk4.T) - pendulum_energy(*state0))
#   err_lf  = ...
#
# Use ax.loglog, and subsample (e.g. [::200]) so the figure stays light.

**TODO A.2** - Overlay the two trajectories in the $(\theta, p)$ plane. One
stays on a closed curve, the other does not. Connect what you see to
Exercise 3.7 on phase-space volume.

In [ ]:
# TODO: phase-plane overlay. Plot the first ~20 periods and the last ~20
# periods of each trajectory in different colours to make the drift visible.

**TODO A.3** - Halve `dt` and confirm the order of each method: the leapfrog
energy *envelope* should shrink by 4, the Runge-Kutta per-step error by 16.
Does the long-term ranking of the two methods change?

In [ ]:
# TODO: loop over dt in [0.1, 0.05, 0.025, 0.0125], measure
# max|H - H0| over a fixed physical time, and fit the slope on log-log axes.

**TODO A.4** - Repeat with an initial condition just inside the separatrix,
$H(0) = 0.999$. Something goes wrong. Explain why it is not the integrator's
fault. *Hint: Exercise 3.8, part 2.*

In [ ]:
# TODO: choose theta0 = 0 and solve p0 from H0 = 0.999, then integrate.
# Watch whether the orbit stays a libration or escapes into circulation.

---
## Part B - Exercise 3.11: rebuilding the phase portrait

Figure 3.2 was drawn from the analytic level curves of $H$. Here you rebuild it
by integration, then use it to measure the period and the action.

**TODO B.1** - Integrate a family of initial conditions and plot their
trajectories in the $(\theta, p)$ plane, distinguishing librations ($H < 1$)
from circulations ($H > 1$). Mark the separatrix, $p = \pm 2|\cos(\theta/2)|$.

In [ ]:
# Suggested families:
#   librations:    theta0 in linspace(0.3, 3.0, 8),  p0 = 0
#   circulations:  theta0 = 0,  p0 in linspace(2.1, 3.2, 4)
#
# TODO: integrate each with leapfrog for one or two periods and plot.

**TODO B.2** - Measure the period $T$ as a function of $H$, and compare with
the analytic result of Exercise 3.8,

$$ T = 4 K(k), \qquad k = \sin(\theta_{\max}/2), \qquad k^2 = \tfrac12 (1 + H). $$

Then verify the logarithmic divergence by plotting $T$ against $-\ln(1 - H)$:
you should find a straight line of slope 2 as $H \to 1$.

`scipy.special.ellipk` takes the parameter $m = k^2$, not the modulus $k$.

In [ ]:
from scipy.special import ellipk


def measure_period(theta0, dt=1e-3, max_periods=1):
    """Return the libration period by detecting the return to theta0 with p > 0.

    A simple bracketing zero-crossing finder is enough here; refine by linear
    interpolation on the crossing so the estimate is not limited by dt.
    """
    raise NotImplementedError("TODO B.2")


# TODO: sweep theta0, measure T, and overlay 4 * ellipk(k**2).

**TODO B.3** - For each libration, compute the enclosed area $2\pi J$ and check
numerically that $\mathrm{d}H/\mathrm{d}J$ equals the measured $2\pi/T$
(Corollary 3.1.1, motion in action-angle variables).

The shoelace formula on the sampled trajectory is accurate enough, provided the
orbit is closed to within a step.

In [ ]:
def enclosed_area(theta, p):
    """Shoelace area of a closed curve sampled at (theta, p)."""
    return 0.5 * np.abs(
        np.sum(theta * np.roll(p, -1) - np.roll(theta, -1) * p)
    )


# TODO: J = enclosed_area(...) / (2 * np.pi) for a family of orbits,
# then compare np.gradient(H_values, J_values) with 2 * np.pi / T_measured.

---
## Part C - Exercise 3.12: frequency ratios and closed orbits

Now leave the pendulum for planar orbits in a spherical potential, where there
are two fundamental frequencies, $\Omega_r$ and $\Omega_\psi$. Example 3.7 of
the notes claims

$$ \Phi \propto -1/r \;\Rightarrow\; \Omega_r : \Omega_\psi = 1:1,
\qquad
\Phi \propto r^2 \;\Rightarrow\; \Omega_r : \Omega_\psi = 2:1 , $$

and that no other spherical potential closes every orbit. Measure the ratio and
see.

In [ ]:
from galdyn.potentials import (
    kepler_dphi_dr,
    harmonic_dphi_dr,
    logarithmic_dphi_dr,
)

# dPhi/dr for each case, in units where the scale is unity.
# galdyn's logarithmic potential is cored (rc = 0.4 by default); set rc = 0
# for the scale-free case discussed in the notes, and try rc > 0 afterwards to
# see how a core pushes the frequency ratio towards the harmonic value of 2.
POTENTIALS = {
    "Kepler": lambda r: kepler_dphi_dr(r, G=1.0, M=1.0),
    "harmonic": lambda r: harmonic_dphi_dr(r, omega0=1.0),
    "logarithmic": lambda r: logarithmic_dphi_dr(r, v0=1.0, rc=0.0),
}

**TODO C.1** - Integrate a planar orbit and record $r(t)$ and $\psi(t)$.
Obtain $\Omega_r$ from the dominant peak of the power spectrum of $r(t)$, and
$\Omega_\psi$ from the mean rate of advance of the unwrapped $\psi(t)$.

Beware of two traps: the FFT resolution is $2\pi / T_{\rm int}$, so integrate
long; and $\psi$ must be unwrapped (`np.unwrap`) before you fit a slope.

In [ ]:
def integrate_planar(dphi_dr, r0, vphi0, dt, n_steps):
    """Integrate a planar orbit in a spherical potential with leapfrog.

    Returns t, r, psi (unwrapped).
    """
    raise NotImplementedError("TODO C.1")


def radial_frequency(t, r):
    """Dominant angular frequency of r(t), from the power spectrum."""
    raise NotImplementedError("TODO C.1")

**TODO C.2** - Run the Kepler and harmonic cases and recover $1{:}1$ and
$2{:}1$. Plot the orbits and confirm they close.

**TODO C.3** - Now the logarithmic potential. Measure
$\Omega_r / \Omega_\psi$ for several orbits, show that it lies strictly
between 1 and 2 and *varies with the orbit*, and integrate one orbit long
enough to watch the rosette fill an annulus.

**TODO C.4** - Plot $\Omega_r / \Omega_\psi$ against eccentricity
$(r_{\rm apo} - r_{\rm peri}) / (r_{\rm apo} + r_{\rm peri})$ for the
logarithmic potential. Is any orbit closed? What would a closed one require of
the integer vector $\mathbf{k}$?

In [ ]:
# TODO: C.2, C.3, C.4.

**TODO C.5 (optional)** - Increase the integration time tenfold and watch the
measured ratio converge. Why is frequency extraction *harder* for a nearly
resonant orbit than for a generic one?

*Hint: how long must you integrate to resolve two frequencies separated by
$\mathbf{k}\cdot\mathbf{\Omega}$?*

In [ ]:
# TODO: C.5.